<a href="https://colab.research.google.com/github/PauloCrot/sudoku-linear-optimization/blob/main/Projeto_Pluma_ERA5_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto integrado: pluma, vento ERA5 e CFD

Este notebook continua o projeto de **detecção e simulação de uma pluma**.

## Objetivos

- acessar um arquivo `.zip` armazenado no Google Drive;
- descompactar os arquivos ERA5 no ambiente temporário do Colab;
- abrir NetCDF com `xarray`;
- converter cada NetCDF para CSV;
- calcular velocidade e direção do vento;
- extrair o vento no ponto da pluma;
- relacionar vento, orientação, comprimento e largura da pluma;
- fornecer parâmetros para um modelo didático de advecção–difusão.

In [1]:
# Instalação das bibliotecas necessárias no Colab
!pip -q install xarray netCDF4

import os, zipfile, glob, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.4 MB/s eta 0:00:00


In [4]:
import glob

# Procura pelo arquivo zip em todo o seu Google Drive
encontrados = glob.glob("/content/drive/MyDrive/**/*modelagem_computacional*.zip", recursive=True)
print("Caminhos encontrados:", encontrados)

Caminhos encontrados: []


In [10]:
# Instala a biblioteca do Copernicus
!pip install -q cdsapi

# 1. Monta o Google Drive para salvar o arquivo lá dentro
from google.colab import drive
drive.mount('/content/drive')

import os
import cdsapi

# Define o caminho exato no seu Google Drive (Tarefa 1)
PASTA_DRIVE = "/content/drive/MyDrive/ModelagemComputacional/projeto_pluma"
os.makedirs(PASTA_DRIVE, exist_ok=True)
CAMINHO_DESTINO = os.path.join(PASTA_DRIVE, "era5_data.nc")

# Configuração da API
clientUrl = "https://cds.climate.copernicus.eu/api"
apiKey = "ba422a31-e3e5-4520-90d8-0a512b45022e" # Sua chave correta

# 2 e 3. Parametrização do pedido (Área, Variáveis e o Dia 27 correto)
dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": "reanalysis",
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_temperature",
        "total_precipitation"
    ],
    "year": "2026",
    "month": ["08"],
    "day": ["27"],  # <--- O dia certo pedido na aula
    "time": [
        "00:00", "03:00", "06:00", "09:00", "12:00", "15:00", "18:00", "21:00"
    ],
    "area": [-23.25, -46.5, -23.5, -46.25], # Delimitação da região
    "format": "netcdf"
}

# Executa o cliente e faz o download direto para o Drive
client = cdsapi.Client(url=clientUrl, key=apiKey)
print(f"Baixando dados do dia 27 para o Drive: {CAMINHO_DESTINO} ...")
client.retrieve(dataset, request).download(CAMINHO_DESTINO)
print("Download concluído com sucesso! O arquivo está seguro no seu Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Baixando dados do dia 27 para o Drive: /content/drive/MyDrive/ModelagemComputacional/projeto_pluma/era5_data.nc ...


2026-09-14 19:42:57,455 INFO Request ID is d0d0ad2c-5c20-49a8-888a-b29a5744bc8f
INFO:ecmwf.datastores.legacy_client:Request ID is d0d0ad2c-5c20-49a8-888a-b29a5744bc8f
2026-09-14 19:42:57,627 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-09-14 19:43:32,317 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-09-14 19:43:49,549 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


aac6555daf0a069212b4a07ea4953458.zip:   0%|          | 0.00/67.4k [00:00<?, ?B/s]

Download concluído com sucesso! O arquivo está seguro no seu Drive.


In [8]:
# Montagem do Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Ajuste este caminho para a pasta onde o ZIP foi salvo
PASTA_DRIVE = "/content/drive/MyDrive/ModelagemComputacional/projeto_pluma"
NOME_ZIP = "modelagem_computacional_era5.zip"
CAMINHO_ZIP = os.path.join(PASTA_DRIVE, NOME_ZIP)

# A extração em /content costuma ser mais rápida durante a aula
PASTA_EXTRACAO = "/content/era5_pluma"
PASTA_SAIDA = os.path.join(PASTA_DRIVE, "csv_era5")
os.makedirs(PASTA_EXTRACAO, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)

assert os.path.exists(CAMINHO_ZIP), f"ZIP não encontrado: {CAMINHO_ZIP}"
with zipfile.ZipFile(CAMINHO_ZIP, "r") as z:
    z.extractall(PASTA_EXTRACAO)

print("Arquivos extraídos:")
for p in sorted(Path(PASTA_EXTRACAO).rglob("*")):
    if p.is_file(): print(p)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivos extraídos:
/content/era5_pluma/10m_u_component_of_wind_stream-oper_daily-mean.nc
/content/era5_pluma/10m_v_component_of_wind_0_daily-mean.nc
/content/era5_pluma/2m_temperature_0_daily-mean.nc


In [16]:
import pandas as pd

arquivos_nc = sorted(glob.glob(f"{PASTA_EXTRACAO}/**/*.nc", recursive=True))
assert arquivos_nc, "Nenhum arquivo NetCDF foi encontrado."

dados_resumo = []
for caminho in arquivos_nc:
    ds = xr.open_dataset(caminho)
    dados_resumo.append({
        "Arquivo": os.path.basename(caminho),
        "Variáveis": ", ".join(list(ds.data_vars)),
        "Dimensões": str(dict(ds.sizes)),
        "Coordenadas": ", ".join(list(ds.coords))
    })
    ds.close()

# Exibe tudo organizado em formato de tabela
df_resumo = pd.DataFrame(dados_resumo)
display(df_resumo)

,Arquivo,Variáveis,Dimensões,Coordenadas
0,10m_u_component_of_wind_stream-oper_daily-mean.nc,u10,"{'valid_time': 1, 'latitude': 3, 'longitude': 3}","number, latitude, longitude, valid_time"
1,10m_v_component_of_wind_0_daily-mean.nc,v10,"{'valid_time': 1, 'latitude': 3, 'longitude': 3}","number, latitude, longitude, valid_time"
2,2m_temperature_0_daily-mean.nc,t2m,"{'valid_time': 1, 'latitude': 3, 'longitude': 3}","number, latitude, longitude, valid_time"


In [15]:
import numpy as np

path_u = [p for p in arquivos_nc if "u_component" in p][0]
path_v = [p for p in arquivos_nc if "v_component" in p][0]

ds_u = xr.open_dataset(path_u)
ds_v = xr.open_dataset(path_v)

u = ds_u['u10']
v = ds_v['v10']

velocidade_vento = np.sqrt(u**2 + v**2)

direcao_vento = (np.rad2deg(np.arctan2(-u, -v)) + 360) % 360

print("Cálculo Realizado com Sucesso")
print("Velocidade média:", float(velocidade_vento.mean()), "m/s")
print("Direção média:", float(direcao_vento.mean()), "graus")

Cálculo Realizado com Sucesso
Velocidade média: 2.189927339553833 m/s
Direção média: 120.3162841796875 graus


In [18]:
# Pega o primeiro ponto de latitude, longitude e tempo para testar
u_val = float(u.isel(latitude=0, longitude=0, valid_time=0))
v_val = float(v.isel(latitude=0, longitude=0, valid_time=0))
ws_val = float(velocidade_vento.isel(latitude=0, longitude=0, valid_time=0))
wd_val = float(direcao_vento.isel(latitude=0, longitude=0, valid_time=0))

# Recálculo manual de prova real
calculo_manual_ws = (u_val**2 + v_val**2)**0.5

print(f"Componente U: {u_val:.2f}")
print(f"Componente V: {v_val:.2f}")
print(f"Velocidade gerada pelo código: {ws_val:.2f}")
print(f"Velocidade calculada manualmente: {calculo_manual_ws:.2f}")
print(f"Direção resultante: {wd_val:.2f}°")

Componente U: -2.29
Componente V: 1.15
Velocidade gerada pelo código: 2.56
Velocidade calculada manualmente: 2.56
Direção resultante: 116.56°


## Conversão de NetCDF para CSV

O NetCDF é eficiente para dados multidimensionais. O CSV é mais simples para inspeção, gráficos e integração com outras etapas do projeto. A conversão usa `to_dataframe()` e preserva as coordenadas como colunas.

In [ ]:
def netcdf_para_csv(caminho_nc, pasta_saida):
    ds = xr.open_dataset(caminho_nc)
    nome = Path(caminho_nc).stem + ".csv"
    caminho_csv = os.path.join(pasta_saida, nome)
    df = ds.to_dataframe().reset_index()
    df.to_csv(caminho_csv, index=False)
    ds.close()
    print(f"{nome}: {df.shape[0]} linhas × {df.shape[1]} colunas")
    return df, caminho_csv

tabelas = {}
for caminho in arquivos_nc:
    chave = Path(caminho).stem
    tabelas[chave], _ = netcdf_para_csv(caminho, PASTA_SAIDA)

print("\nCSV salvos em:", PASTA_SAIDA)
print(os.listdir(PASTA_SAIDA))

In [ ]:
# Conferência da tabela e normalização dos nomes
df_exemplo = next(iter(tabelas.values())).copy()
print(df_exemplo.head())
print(df_exemplo.dtypes)

def encontrar_coluna(df, candidatos):
    for c in candidatos:
        if c in df.columns: return c
    raise KeyError(f"Nenhuma destas colunas foi encontrada: {candidatos}. Colunas: {list(df.columns)}")

COL_LAT = encontrar_coluna(df_exemplo, ["latitude", "lat"])
COL_LON = encontrar_coluna(df_exemplo, ["longitude", "lon"])
COL_TIME = next((c for c in ["time", "valid_time", "date"] if c in df_exemplo.columns), None)
print(COL_LAT, COL_LON, COL_TIME)

## Coordenadas da pluma e extração do vento

Substitua os valores abaixo pelas coordenadas usadas na primeira aula. Como ERA5 possui uma grade relativamente espaçada, a seleção mais simples é o ponto de grade mais próximo. Para uma análise mais refinada, pode-se usar interpolação bilinear.

In [ ]:
# Exemplo: substitua pelas coordenadas reais da pluma
LAT_PLUMA = -23.05
LON_PLUMA = -45.35

# Localizar automaticamente os CSVs de u e v pelo nome do arquivo
def achar_csv(fragmento):
    encontrados = [p for p in glob.glob(f"{PASTA_SAIDA}/*.csv") if fragmento.lower() in os.path.basename(p).lower()]
    if not encontrados: raise FileNotFoundError(f"CSV com '{fragmento}' não encontrado")
    return encontrados[0]

ARQ_U = achar_csv("10m_u_component")
ARQ_V = achar_csv("10m_v_component")
dfu = pd.read_csv(ARQ_U)
dfv = pd.read_csv(ARQ_V)

COL_U = [c for c in dfu.columns if c not in [COL_LAT, COL_LON, COL_TIME]][0]
COL_V = [c for c in dfv.columns if c not in [COL_LAT, COL_LON, COL_TIME]][0]

# Grade mais próxima para u e v
def ponto_mais_proximo(df, lat, lon):
    idx = ((df[COL_LAT]-lat)**2 + (df[COL_LON]-lon)**2).idxmin()
    return df.loc[idx]

pu = ponto_mais_proximo(dfu, LAT_PLUMA, LON_PLUMA)
pv = ponto_mais_proximo(dfv, LAT_PLUMA, LON_PLUMA)
u10 = float(pu[COL_U]); v10 = float(pv[COL_V])
velocidade = np.hypot(u10, v10)
direcao_vetor = (np.degrees(np.arctan2(v10, u10)) + 360) % 360

print(f"Ponto ERA5 mais próximo: lat={pu[COL_LAT]:.3f}, lon={pu[COL_LON]:.3f}")
print(f"u10={u10:.3f} m/s; v10={v10:.3f} m/s")
print(f"Velocidade={velocidade:.3f} m/s")
print(f"Ângulo do vetor, a partir do eixo leste: {direcao_vetor:.1f}°")
print("Atenção: direção meteorológica 'de onde vem' exige outra convenção e não deve ser confundida com o ângulo do vetor.")

In [ ]:
# Combinar u e v por latitude, longitude e tempo
chaves = [c for c in [COL_TIME, COL_LAT, COL_LON] if c is not None and c in dfu.columns and c in dfv.columns]
df_vento = dfu[chaves + [COL_U]].merge(dfv[chaves + [COL_V]], on=chaves, how="inner")
df_vento["velocidade_10m_m_s"] = np.hypot(df_vento[COL_U], df_vento[COL_V])
df_vento["angulo_vetor_graus"] = (np.degrees(np.arctan2(df_vento[COL_V], df_vento[COL_U])) + 360) % 360
df_vento.to_csv(os.path.join(PASTA_SAIDA, "vento_era5_combinado.csv"), index=False)
df_vento.head()

In [ ]:
# Visualização do campo de vento na grade
fig, ax = plt.subplots(figsize=(9,6))
recorte = df_vento
sc = ax.scatter(recorte[COL_LON], recorte[COL_LAT], c=recorte["velocidade_10m_m_s"], cmap="viridis", s=45)
ax.quiver(recorte[COL_LON], recorte[COL_LAT], recorte[COL_U], recorte[COL_V], color="white", scale=35)
ax.scatter(LON_PLUMA, LAT_PLUMA, marker="*", s=180, c="red", label="pluma")
fig.colorbar(sc, ax=ax, label="Velocidade do vento (m/s)")
ax.set(xlabel="Longitude", ylabel="Latitude", title="ERA5: campo de vento a 10 m")
ax.legend(); plt.show()

## Variáveis que podem ser correlacionadas

A correlação só é interpretável se as grandezas forem comparadas na mesma escala espacial e temporal. Para cada imagem ou vídeo, registre uma tabela de observações com: comprimento da pluma, área da máscara YOLO, largura média, centroide, orientação do eixo principal, posição da fonte, data e hora.

Sugestões: (1) comprimento versus velocidade do vento; (2) orientação da pluma versus ângulo do vetor do vento; (3) largura versus difusividade efetiva; (4) deslocamento do centroide versus velocidade integrada no tempo; (5) área/IoU da máscara versus área da região simulada acima de um limiar de concentração.

In [ ]:
# Exemplo de tabela para medições extraídas da imagem ou do YOLO
observacoes = pd.DataFrame({
    "data_hora": [], "comprimento_m": [], "largura_media_m": [],
    "area_pluma_m2": [], "angulo_pluma_graus": [],
    "centroide_x_m": [], "centroide_y_m": []
})

# Depois de preencher as observações, juntar ao vento pelo instante mais próximo:
# observacoes["data_hora"] = pd.to_datetime(observacoes["data_hora"])
# df_vento["time"] = pd.to_datetime(df_vento["time"])
# analise = pd.merge_asof(observacoes.sort_values("data_hora"),
#     df_vento.sort_values("time"), left_on="data_hora", right_on="time",
#     direction="nearest", tolerance=pd.Timedelta("3h"))
# analise.corr(numeric_only=True)[["velocidade_10m_m_s", "angulo_vetor_graus"]]]